# Expt 9 - Text Classification and Text Generation using Transformers
Name: K ADHITHYA

Roll No: 24BAD004

### Part A: Dataset preparation

In [1]:
!pip install -q transformers datasets accelerate scikit-learn sentencepiece

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer

print("Roll No: 24BAD009")

dataset = load_dataset("stanfordnlp/imdb")
print(dataset)

for i in range(3):
    print(dataset["train"][i]["label"], dataset["train"][i]["text"][:200])
    print()

print(pd.Series(dataset["train"]["label"]).value_counts())
print(pd.Series(dataset["test"]["label"]).value_counts())

Roll No: 24BAD009
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
0 I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev

0 "I Am Curious: Yellow" is a risible and pretentious steaming pile. It doesn't matter what one's political views are because this film can hardly be taken seriously on any level. As for the claim that 

0 If only to avoid making this type of film in the future. This film is interesting as an experiment but tells no cogent story.<br /><br />One might feel virtuous for sitting thru it because it touches 

0    12500
1    12500
Name: count, dtype: int64
0    12500
1    12500
Name: coun

In [3]:
train_data = dataset["train"].shuffle(seed=42).select(range(2000))
test_data = dataset["test"].shuffle(seed=42).select(range(500))

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=256)

train_tok = train_data.map(tokenize, batched=True)
test_tok = test_data.map(tokenize, batched=True)

train_tok = train_tok.remove_columns(["text"]).rename_column("label", "labels")
test_tok = test_tok.remove_columns(["text"]).rename_column("label", "labels")
train_tok.set_format("torch")
test_tok.set_format("torch")

print(train_tok)
print(train_tok[0]["input_ids"].shape)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2000
})
torch.Size([256])


In [4]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_tok, batch_size=16, shuffle=True)
test_loader = DataLoader(test_tok, batch_size=32)

batch = next(iter(train_loader))
print(batch["input_ids"].shape, batch["attention_mask"].shape, batch["labels"].shape)

torch.Size([16, 256]) torch.Size([16, 256]) torch.Size([16])


### Part B: Fine-tuning BERT

In [6]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print("Roll No: 24BAD009")

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    p, r, f, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": accuracy_score(labels, preds), "precision": p, "recall": r, "f1": f}

import inspect

training_args_params = inspect.signature(TrainingArguments.__init__).parameters
evaluation_key = "eval_strategy" if "eval_strategy" in training_args_params else "evaluation_strategy"

training_args_config = {
    "output_dir": "bert_imdb",
    "learning_rate": 2e-5,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 32,
    "num_train_epochs": 2,
    "save_strategy": "no",
    "logging_steps": 50,
    "report_to": "none"
}
training_args_config[evaluation_key] = "epoch"

args = TrainingArguments(**training_args_config)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    compute_metrics=compute_metrics
)

trainer.train()

Roll No: 24BAD009


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.425506,0.297703,0.870000,0.836431,0.914634,0.873786
2,0.198446,0.286561,0.884000,0.864341,0.906504,0.884921


TrainOutput(global_step=250, training_loss=0.3315768394470215, metrics={'train_runtime': 186.061, 'train_samples_per_second': 21.498, 'train_steps_per_second': 1.344, 'total_flos': 526222110720000.0, 'train_loss': 0.3315768394470215, 'epoch': 2.0})

In [7]:
results = trainer.evaluate()
print(results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.198446,0.286561,2,0.884000,0.864341,0.906504,0.884921


{'eval_loss': 0.2865613102912903, 'eval_accuracy': 0.884, 'eval_precision': 0.8643410852713178, 'eval_recall': 0.9065040650406504, 'eval_f1': 0.8849206349206349}


In [8]:
import torch

def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
    label = logits.argmax(-1).item()
    return "Positive" if label == 1 else "Negative"

new_texts = [
    "This movie was amazing, I loved every minute of it.",
    "Terrible plot and bad acting. Complete waste of time.",
    "The story was okay but the ending was disappointing."
]
for t in new_texts:
    print(t, "->", predict(t))

model.save_pretrained("bert_sentiment_model")
tokenizer.save_pretrained("bert_sentiment_model")

This movie was amazing, I loved every minute of it. -> Positive
Terrible plot and bad acting. Complete waste of time. -> Negative
The story was okay but the ending was disappointing. -> Negative


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('bert_sentiment_model/tokenizer_config.json',
 'bert_sentiment_model/tokenizer.json')

### Part C: Sequence-to-sequence with T5

In [9]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

print("Roll No: 24BAD009")

t5_tokenizer = T5Tokenizer.from_pretrained("t5-small")
t5_model = T5ForConditionalGeneration.from_pretrained("t5-small")

text = """The government announced a new policy to promote renewable energy across the country.
Under the plan, solar and wind projects will receive subsidies and tax benefits. Officials
said the move will reduce carbon emissions and create thousands of new jobs over the next decade."""

def generate(prompt, max_len=50, beams=4):
    ids = t5_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).input_ids
    out = t5_model.generate(ids, max_length=max_len, num_beams=beams, early_stopping=True)
    return t5_tokenizer.decode(out[0], skip_special_tokens=True)

print("Input:", text)
print()
print("Summary:", generate("summarize: " + text))

Roll No: 24BAD009


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Input: The government announced a new policy to promote renewable energy across the country.
Under the plan, solar and wind projects will receive subsidies and tax benefits. Officials
said the move will reduce carbon emissions and create thousands of new jobs over the next decade.

Summary: the government announced a new policy to promote renewable energy across the country. under the plan, solar and wind projects will receive subsidies and tax benefits.


In [16]:
rows = []
for max_len in [20, 40, 80]:
    for beams in [1, 4]:
        out = generate("summarize: " + text, max_len, beams)
        rows.append({"max_length": max_len, "beams": beams, "output": out})
        print(max_len, beams, "->", out)

pd.DataFrame(rows).to_csv("t5_outputs.csv", index=False)

20 1 -> the government announced a new policy to promote renewable energy across the country. the plan will
20 4 -> the government announced a new policy to promote renewable energy across the country. under the plan
40 1 -> the government announced a new policy to promote renewable energy across the country. the plan will reduce carbon emissions and create thousands of new jobs.
40 4 -> the government announced a new policy to promote renewable energy across the country. under the plan, solar and wind projects will receive subsidies and tax benefits.
80 1 -> the government announced a new policy to promote renewable energy across the country. the plan will reduce carbon emissions and create thousands of new jobs.
80 4 -> the government announced a new policy to promote renewable energy across the country. under the plan, solar and wind projects will receive subsidies and tax benefits.


In [11]:
print(generate("translate English to German: How are you today?"))
print(generate("translate English to French: The weather is very nice today."))

Wie sind Sie heute?
Le temps est très agréable aujourd'hui.


### Part D: Testing and analysis

In [ ]:
print("Roll No: 24BAD009")

tests = [
    "An excellent film with brilliant performances.",
    "I hated this movie, it was boring and too long.",
    "Great visuals but the script was weak.",
    "Not bad at all, actually quite enjoyable."
]
for t in tests:
    print(predict(t), "|", t)

In [ ]:
output = trainer.predict(test_tok)
preds = output.predictions.argmax(-1)

from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(output.label_ids, preds, target_names=["Negative", "Positive"]))
print(confusion_matrix(output.label_ids, preds))

In [14]:
wrong = np.where(preds != output.label_ids)[0]
print("Misclassified:", len(wrong), "out of", len(preds))

for i in wrong[:3]:
    print("Actual:", output.label_ids[i], "Predicted:", preds[i])
    print(test_data[int(i)]["text"][:300])
    print()

Misclassified: 58 out of 500
Actual: 0 Predicted: 1
Coming from Kiarostami, this art-house visual and sound exposition is a surprise. For a director known for his narratives and keen observation of humans, especially children, this excursion into minimalist cinematography begs for questions: Why did he do it? Was it to keep him busy during a vacation

Actual: 0 Predicted: 1
Intended as light entertainment, this film is indeed successful as such during its first half, but then succumbs to a rapidly foundering script that drops it down. Harry (Judd Nelson), a "reformed" burglar, and Daphne (Gina Gershon), an aspiring actress, are employed as live window mannequins at a d

Actual: 0 Predicted: 1
"An astronaut (Michael Emmet) dies while returning from a mission and his body is recovered by the military. The base where the dead astronaut is taken to becomes the scene of a bizarre invasion plan from outer space. Alien embryos inside the dead astronaut resurrect the corpse and begin a terrify

In [15]:
texts = [
    "Artificial intelligence is changing healthcare by helping doctors detect diseases earlier. Hospitals are using machine learning to analyse scans and predict patient risks. Experts say this will improve treatment and lower costs.",
    "The football match ended in a draw after both teams missed several chances. The coach said he was proud of the players despite the result."
]
for t in texts:
    print("Input:", t)
    print("Short:", generate("summarize: " + t, 25, 4))
    print("Long:", generate("summarize: " + t, 60, 4))
    print()

Input: Artificial intelligence is changing healthcare by helping doctors detect diseases earlier. Hospitals are using machine learning to analyse scans and predict patient risks. Experts say this will improve treatment and lower costs.
Short: artificial intelligence is changing healthcare by helping doctors detect diseases earlier. hospitals are using machine learning to analyse scans and predict
Long: hospitals are using machine learning to analyse scans and predict patient risks. this will improve treatment and lower costs.

Input: The football match ended in a draw after both teams missed several chances. The coach said he was proud of the players despite the result.
Short: the match ended in a draw after both teams missed several chances. the coach said he was proud of the
Long: the match ended in a draw after both teams missed several chances. coach said he was proud of the players despite the result.

